# 11. BLAST: Searching Every Known Sequence

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `seeds.py`

BLAST's first step by hand: seed words shared by human beta and alpha globin, scored with BLOSUM62.


In [ ]:
# BLAST's first step by hand: seed words shared by human beta and alpha
# globin, scored with BLOSUM62.
from Bio import SeqIO
from Bio.Align import substitution_matrices

WORD = 3  # BLAST's word length for proteins
THRESHOLD = 11  # a pair of words must score at least this to seed

blosum62 = substitution_matrices.load("BLOSUM62")
beta = str(SeqIO.read("hbb.fasta", "fasta").seq)
alpha = str(SeqIO.read("hba_human.fasta", "fasta").seq)


def word_score(a: str, b: str) -> float:
    return sum(blosum62[x][y] for x, y in zip(a, b))


seeds = []
for i in range(len(beta) - WORD + 1):
    for j in range(len(alpha) - WORD + 1):
        score = word_score(beta[i : i + WORD], alpha[j : j + WORD])
        if score >= THRESHOLD:
            seeds.append((i, j, score))

pairs = (len(beta) - WORD + 1) * (len(alpha) - WORD + 1)
print(f"{pairs:,} pairs of words compared, {len(seeds)} seeds")
# The true alignment (Chapter 10) keeps i and j within a few positions
on_diagonal = [s for s in seeds if abs(s[0] - s[1]) <= 8]
print(
    f"{len(on_diagonal)} of them lie within 8 positions of the diagonal"
)
for i, j, score in on_diagonal[:8]:
    print(
        f"  beta {i + 1:3} {beta[i : i + WORD]}   alpha {j + 1:3} "
        f"{alpha[j : j + WORD]}   score {score:4.0f}"
    )

## `remote_blast.py`

BLAST human beta globin against Swiss-Prot on NCBI's servers. The search runs once; the result is saved and read from the file afterwards.


In [ ]:
# BLAST human beta globin against Swiss-Prot on NCBI's servers. The search
# runs once; the result is saved and read from the file afterwards.
from pathlib import Path

from Bio import Blast, SeqIO

Blast.email = "your.name@example.org"  # NCBI asks for a real address
SAVED = Path("hbb_swissprot.xml")

if not SAVED.exists():
    beta = SeqIO.read("hbb.fasta", "fasta").seq
    print("submitting to NCBI; this usually takes a few minutes")
    with Blast.qblast(
        "blastp", "swissprot", beta, hitlist_size=50
    ) as answer:
        SAVED.write_bytes(answer.read())

with SAVED.open("rb") as handle:
    record = Blast.read(handle)

print(f"{len(record)} hits from {record.db}")
for hit in record[:12]:
    best = hit[0]
    species = hit.target.description.split("[")[-1].rstrip("]")
    print(
        f"  {hit.target.id:14} {best.annotations['evalue']:9.1e}"
        f"  {species[:40]}"
    )

## `hits.py`

Read BLAST's tabular output (-outfmt 6), keeping the trustworthy hits.


In [ ]:
# Read BLAST's tabular output (-outfmt 6), keeping the trustworthy hits.
import pandas as pd
from Bio import SeqIO

COLUMNS = [
    "query",
    "subject",
    "identity",
    "length",
    "mismatches",
    "gap_opens",
    "q_start",
    "q_end",
    "s_start",
    "s_end",
    "evalue",
    "bits",
]
CUTOFF = 1e-3

hits = pd.read_csv("hits.tsv", sep="\t", names=COLUMNS)
print(
    f"{len(hits)} hits reported, {(hits['evalue'] < CUTOFF).sum()} "
    f"with an E-value below {CUTOFF}"
)

names = SeqIO.index("human_proteins.fasta", "fasta")
good = hits[hits["evalue"] < CUTOFF].copy()
good["protein"] = [
    names[s].description.split(" ", 1)[1].split(" OS=")[0][:28]
    for s in good["subject"]
]
good["entry"] = good["subject"].str.split("|").str[2]
columns = ["entry", "protein", "identity", "length", "evalue", "bits"]
print(
    good[columns].to_string(
        index=False, formatters={"evalue": "{:.1e}".format}
    )
)